In [1]:
# Chain:
# The steps are predictable and always the same
# You're doing RAG, summarization, classification, extraction

# Agents:
# The steps needed depend on the input or intermediate results.
# Use when you're building something like a chatbot that can do things, not just answer questions

In [7]:
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()

result = search.invoke("current population of Japan.")
print(result)
print(len(result))


Japanese has around 128 million speakers, primarily in Japan, the only country where it is the national language, and within the Japanese diaspora across the globe. The sex ratio in Japan in 2021 was 95.38 males per 100 females. There are 61.53 million males and 64.52 million females in Japan. Population of Japan: current, historical, and projected population, growth rate, immigration, median age, total fertility rate (TFR), population density, urbanization, urban population, country's share of world population, and global rank. Data tables, maps, charts, and live population clock 3 days ago · It has a population of 122.4 million, making it the 12th largest country in the world. Its capital is Tokyo. Japan has a advanced economy with strong technology and manufacturing sectors. Oct 1, 2024 · The total population was 123,802 thousand, a decrease of 550 thousand compared with the previous year. The rate of decrease was 0.44 percent. The total population decreased for the fourteenth year 

In [4]:
from langchain_community.tools import tool

@tool
def get_word_length(output:str)-> int:
    """ Return the length of the output.
    Use this when you need to know how long the output is.
    """
    return int(len(output))

print(get_word_length.name)
print(get_word_length.description)
print(get_word_length.args)

print(get_word_length.invoke("hello"))



get_word_length
Return the length of the output.
Use this when you need to know how long the output is.
{'output': {'title': 'Output', 'type': 'string'}}
5


In [9]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def get_current_datetime(format: str = "%Y-%m-%d %H:%M:%S"):
    """Returns the current date and time.
    Use this when the user asks about the current time or date.
    The format parameter is optional and follows Python strftime conventions.

    """

    return datetime.now().strftime(format)

print(get_current_datetime.invoke({}))

2026-05-25 22:53:32


In [16]:
tools = [get_current_datetime, get_word_length]
for t in tools:
    print(t.name)
    print(t.description)
    print(t.args)
    print("-"* 84)

get_current_datetime
Returns the current date and time.
Use this when the user asks about the current time or date.
The format parameter is optional and follows Python strftime conventions.
{'format': {'default': '%Y-%m-%d %H:%M:%S', 'title': 'Format', 'type': 'string'}}
------------------------------------------------------------------------------------
get_word_length
Return the length of the output.
Use this when you need to know how long the output is.
{'output': {'title': 'Output', 'type': 'string'}}
------------------------------------------------------------------------------------


In [18]:
from dotenv import load_dotenv

load_dotenv()


True

In [21]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

# Tools
@tool
def get_word_length(word: str) -> int:
    """Returns the number of characters in a word.
    Use this when you need to find out how long a word is."""
    return len(word)

@tool
def add_numbers(a: float, b: float) -> float:
    """Adds two numbers together.
    Use this when you need to perform addition."""
    return a + b

tools = [get_word_length, add_numbers]

# LLM
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")

# Create agent — returns a compiled LangGraph directly
agent = create_agent(model=llm, tools=tools)

# Invoke
result = agent.invoke({"messages": [{"role": "user", "content": "What is the length of the word 'intelligence'?"}]})

# The final response is the last message
print(result)

{'messages': [HumanMessage(content="What is the length of the word 'intelligence'?", additional_kwargs={}, response_metadata={}, id='769eca1f-bd56-4c64-bf26-fbf93e2a5763'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_word_length', 'arguments': '{"word": "intelligence"}'}, '__gemini_function_call_thought_signatures__': {'5fda909a-c4cd-4d92-ac5b-d32c1a5d85a3': 'EsYCCsMCAQw51sf3h99G+b7vNe9QUjx2MxZli0DaYVYJV2AaOUOq7OT+gesqvDtNs+geUWOFOcYQJiFRK1r1RbTGmSB9BI1DSfIlF8d2JvKXSvyjs2eW6rsRNiqrf5J/qEVlEFW+0atwnRI6U8nT+z1ugvf6Jim6xfQBiq9saj9yYSxwz/4enSE5nkfoJcTysaoeI5IK7lpORj6RMleKTdkdKTjoVchEdp6Lrd0QvZdmpKVZnihTMZdVIYQ3UDPUxl7+0KUXJde+1xjDEBcbjGB1/zL1jw5ONTo8rYyF5iicU1NUZat+4xf+jAjG1/MXwMs9yGhx56v6KhzmJIRbs6upGXb9BuegM4qy7/iTmluWIZMmjHaOiCDgHoFxaIMLymbmkLGiInp3vgphQYiLz6Q8ugbG/oYJu1+s6kidFz+jD6/wBj6FBpg='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e603e-64a7-7542

In [24]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the length of the word 'intelligence'?"}]},
    stream_mode = "values"
):
    # for node_name, node_output in chunk.items():
    #     print(f"\n--- {node_name} ---")
    #     for message in node_output.get("messages", []):
    #         print(f"[{message.type}]: {message.content}")
    #         if hasattr(message, "tool_calls") and message.tool_calls:
    #             print(f"Tool calls: {message.tool_calls}")
    print(chunk)

{'messages': [HumanMessage(content="What is the length of the word 'intelligence'?", additional_kwargs={}, response_metadata={}, id='edbed257-6c13-4ef5-a9e5-3ed6321cf368')]}
{'messages': [HumanMessage(content="What is the length of the word 'intelligence'?", additional_kwargs={}, response_metadata={}, id='edbed257-6c13-4ef5-a9e5-3ed6321cf368'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_word_length', 'arguments': '{"word": "intelligence"}'}, '__gemini_function_call_thought_signatures__': {'41cc344b-a7ce-42ef-97eb-b079401aa06f': 'EvACCu0CAQw51sd2KnjuuMKMk9q1UdjpGOVpinr7wbyP2UJQZAEmWbvx96/QTH+6FC1XOqkx67jTr7Es7blqtrxq2lzj3e8lVQehE0sCt1Iga3wCR7BsajE8W4wWTY5U6bdYUFcPP78Cpc+PS4wHXyrDoco/dOZ3HMMa1XxP21+CgRqbmQf3qS61ExpsHBRM38oZOB0KrFH8Mh8U6FP7xGkQC7spn4aMwgu794gdXq1ouzfJj7T3XnCYEP7TY54KsvDgKyqZtzl2PP/4ekA8Nf9HotGHG0gitPasneiB6kvU0OG+2kjhqg71UJeYBwHAKqjwcmD8Ap7i/cUH/xdU6YNqeNpABMRIMPfX7EFL/AaA35K9IgVe8qlDTIEHdCUgVIomM6ksTG+iMLS3wT5gg3sR9YGf8ODiFYPw0pJPDfdsb70eqKt4

In [1]:
# wraps tool errors using a middleware.
# this returns a tool message to the LLM that the tool has failed to produce the result,
# based on which the LLM can decide if to tell the user that the use a different input or the tool has failed.

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

@tool
def get_word_length(word: str) -> int:
    """Returns the number of characters in a word."""
    return len(word)

@tool
def divide_numbers(a: float, b: float) -> float:
    """Divides a by b."""
    return a / b

@wrap_tool_call
def handle_tool_errors(request, handler):
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool failed: {str(e)}. Try a different approach.",
            tool_call_id=request.tool_call["id"]
        )

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key="YOUR_API_KEY")

agent = create_agent(
    model=llm,
    tools=[get_word_length, divide_numbers],
    system_prompt="You are a helpful assistant.",
    middleware=[handle_tool_errors]
)

# Stream to see every step
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the length of 'hello' and what is 10 divided by 0?"}]},
    stream_mode="updates"
):
    for node_name, node_output in chunk.items():
        print(f"\n--- {node_name} ---")
        for message in node_output.get("messages", []):
            print(f"[{message.type}]: {message.content}")
            if hasattr(message, "tool_calls") and message.tool_calls:
                print(f"Tool calls: {message.tool_calls}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}